# SVM Classification

This notebook trains an SVM classifier to predict whether an engine is likely to fail soon.

The same feature engineering, failure definition, engine-wise train-validation split, and cost-sensitive threshold selection
used in the previous models are retained for fair comparison.

In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [3]:
import pandas as pd
import numpy as np

columns = (
    ["engine_id", "cycle"]
    + ["setting_1", "setting_2", "setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train = pd.read_csv(
    "CMAPSSData/train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

# Create RUL
train["RUL"] = (
    train.groupby("engine_id")["cycle"]
    .transform("max") - train["cycle"]
)

train.head()

,engine_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187


In [4]:
constant_sensors = [
    "sensor_1",
    "sensor_5",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
]

train_clean = train.drop(columns=constant_sensors)

In [5]:
feature_sensors = [
    col for col in train_clean.columns
    if col.startswith("sensor_")
]

feature_data = train_clean.copy()

for sensor in feature_sensors:

    # Rolling mean
    feature_data[f"{sensor}_rolling_mean"] = (
        feature_data.groupby("engine_id")[sensor]
        .transform(
            lambda x: x.rolling(window=5, min_periods=1).mean()
        )
    )

    # Rolling standard deviation
    feature_data[f"{sensor}_rolling_std"] = (
        feature_data.groupby("engine_id")[sensor]
        .transform(
            lambda x: x.rolling(window=5, min_periods=1).std()
        )
    )

    # Change from previous cycle
    feature_data[f"{sensor}_delta"] = (
        feature_data.groupby("engine_id")[sensor].diff()
    )

In [6]:
failure_threshold = 30

feature_data["failure_soon"] = (
    feature_data["RUL"] <= failure_threshold
).astype(int)

print(feature_data["failure_soon"].value_counts())

failure_soon
0    17531
1     3100
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

engine_ids = feature_data["engine_id"].unique()

train_engines, val_engines = train_test_split(
    engine_ids,
    test_size=0.20,
    random_state=42
)

train_data = feature_data[
    feature_data["engine_id"].isin(train_engines)
].copy()

val_data = feature_data[
    feature_data["engine_id"].isin(val_engines)
].copy()

print("Training engines:", train_data["engine_id"].nunique())
print("Validation engines:", val_data["engine_id"].nunique())

print("Training shape:", train_data.shape)
print("Validation shape:", val_data.shape)

Training engines: 80
Validation engines: 20
Training shape: (16561, 67)
Validation shape: (4070, 67)


In [8]:
X_train = train_data.drop(
    columns=["engine_id", "RUL", "failure_soon"]
)

y_train = train_data["failure_soon"]

X_val = val_data.drop(
    columns=["engine_id", "RUL", "failure_soon"]
)

y_val = val_data["failure_soon"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

X_train shape: (16561, 64)
y_train shape: (16561,)
X_val shape: (4070, 64)
y_val shape: (4070,)


In [9]:
imputer = SimpleImputer(strategy="median")

In [10]:
# Fit imputer on trainig data
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

print("X_train imputed shape:", X_train_imputed.shape)
print("X_val imputed shape:", X_val_imputed.shape)

X_train imputed shape: (16561, 64)
X_val imputed shape: (4070, 64)


In [11]:
# Scale the features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)

print("X_train scaled shape:", X_train_scaled.shape)
print("X_val scaled shape:", X_val_scaled.shape)

X_train scaled shape: (16561, 64)
X_val scaled shape: (4070, 64)


In [12]:
# Train SVM
svm_clf = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42
)

svm_clf.fit(X_train_scaled, y_train)

print("SVM training completed.")

C:\Users\avnis\anaconda3_newml\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM training completed.


In [13]:
# Predict failure probabilities

y_prob_svm = svm_clf.predict_proba(X_val_scaled)[:, 1]

print("First 20 failure probabilities:")
print(y_prob_svm[:20])

First 20 failure probabilities:
[6.88611251e-06 1.33076653e-03 2.02753187e-04 1.03078396e-04
 2.02722590e-05 7.69874138e-05 6.04705871e-05 1.42507553e-04
 9.42479218e-05 1.14697647e-04 3.41223834e-04 1.16077581e-04
 5.73920926e-04 2.93979887e-03 2.90723604e-04 1.36204935e-04
 2.42059196e-05 1.41792894e-05 3.60865221e-05 2.15609196e-04]


In [14]:
# test different thresholds
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.10, 0.91, 0.05)

results_svm = []

for threshold in thresholds:
    y_pred = (y_prob_svm >= threshold).astype(int)

    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    results_svm.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    })

threshold_df_svm = pd.DataFrame(results_svm)

threshold_df_svm

,threshold,precision,recall,f1_score
0,0.10,0.663366,0.972581,0.788751
1,0.15,0.703878,0.966129,0.814412
2,0.20,0.729963,0.954839,0.827393
3,0.25,0.744332,0.953226,0.835926
4,0.30,0.765319,0.946774,0.846431
5,0.35,0.792402,0.941935,0.860722
6,0.40,0.802503,0.930645,0.861837
7,0.45,0.817266,0.916129,0.863878
8,0.50,0.831343,0.898387,0.863566
9,0.55,0.838415,0.887097,0.862069


In [16]:
from sklearn.metrics import confusion_matrix

In [17]:
# Cost-sensitive threshold optimization

FP_COST = 1
FN_COST = 5

cost_results_svm = []

for threshold in thresholds:
    y_pred = (y_prob_svm >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    total_cost = (FP_COST * fp) + (FN_COST * fn)

    cost_results_svm.append({
        "threshold": threshold,
        "false_positives": fp,
        "false_negatives": fn,
        "total_cost": total_cost
    })

cost_df_svm = pd.DataFrame(cost_results_svm)

cost_df_svm

,threshold,false_positives,false_negatives,total_cost
0,0.10,306,17,391
1,0.15,252,21,357
2,0.20,219,28,359
3,0.25,203,29,348
4,0.30,180,33,345
5,0.35,153,36,333
6,0.40,142,43,357
7,0.45,127,52,387
8,0.50,113,63,428
9,0.55,106,70,456


In [18]:
best_threshold_svm = cost_df_svm.loc[
    cost_df_svm["total_cost"].idxmin(),
    "threshold"
]

min_cost_svm = cost_df_svm["total_cost"].min()

print("Best cost-sensitive threshold:", best_threshold_svm)
print("Minimum total cost:", min_cost_svm)

Best cost-sensitive threshold: 0.3500000000000001
Minimum total cost: 333


In [19]:
y_pred_svm_final = (
    y_prob_svm >= best_threshold_svm
).astype(int)

In [20]:
from sklearn.metrics import confusion_matrix, classification_report

cm_svm_final = confusion_matrix(y_val, y_pred_svm_final)

print("Final SVM Confusion Matrix:")
print(cm_svm_final)

print("\nFinal SVM Classification Report:")
print(classification_report(y_val, y_pred_svm_final))

Final SVM Confusion Matrix:
[[3297  153]
 [  36  584]]

Final SVM Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.97      3450
           1       0.79      0.94      0.86       620

    accuracy                           0.95      4070
   macro avg       0.89      0.95      0.92      4070
weighted avg       0.96      0.95      0.96      4070

